# Preprocessing - Heart Disease Dataset

PPreprocessing sau EDA.

- Đọc dữ liệu raw từ `data/raw/heart.csv`.
- Áp dụng các quyết định làm sạch đã rút ra từ EDA.
- Chuyển target gốc `0, 1, 2, 3, 4` thành binary target.
- Chia train/test có stratify theo target.
- Fit scaler/encoder chỉ trên train để tránh data leakage.
- Lưu dữ liệu đã xử lý vào `data/processed/` để dùng cho bước training.


In [ ]:
# Import library
from pathlib import Path
import json

import joblib
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Set PATHPATH
PROJECT_ROOT = Path("..").resolve()
RAW_DATA_PATH = PROJECT_ROOT / "data" / "raw" / "heart.csv"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
MODELS_DIR = PROJECT_ROOT / "models"

# Set seed
RANDOM_STATE = 42
TEST_SIZE = 0.2

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Raw data path: {RAW_DATA_PATH}")
print(f"Processed data will be saved to: {PROCESSED_DIR}")

Raw data path: /home/dthanh/Documents/Personal_Doc/ThS/HP2/PPNCKH/Project/heart-disease-classification/data/raw/heart.csv
Processed data will be saved to: /home/dthanh/Documents/Personal_Doc/ThS/HP2/PPNCKH/Project/heart-disease-classification/data/processed


## 1. Load Raw Data


In [ ]:
columns = [
    "age",
    "sex",
    "cp",
    "trestbps",
    "chol",
    "fbs",
    "restecg",
    "thalach",
    "exang",
    "oldpeak",
    "slope",
    "ca",
    "thal",
    "target",
]

# Read raw data
df_raw = pd.read_csv(RAW_DATA_PATH, names=columns, na_values="?")

display(df_raw.head())
print(f"Raw shape: {df_raw.shape}")

,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,target
0,63.0,1.0,1.0,145.0,233.0,1.0,2.0,150.0,0.0,2.3,3.0,0.0,6.0,0
1,67.0,1.0,4.0,160.0,286.0,0.0,2.0,108.0,1.0,1.5,2.0,3.0,3.0,2
2,67.0,1.0,4.0,120.0,229.0,0.0,2.0,129.0,1.0,2.6,2.0,2.0,7.0,1
3,37.0,1.0,3.0,130.0,250.0,0.0,0.0,187.0,0.0,3.5,3.0,0.0,3.0,0
4,41.0,0.0,2.0,130.0,204.0,0.0,2.0,172.0,0.0,1.4,1.0,0.0,3.0,0


Raw shape: (303, 14)


## 2. Define Feature Groups


In [ ]:
# Numeric features scale by StandardScaler
numeric_features = ["age", "trestbps", "chol", "thalach", "oldpeak"]

# Categorical features
categorical_features = ["sex", "cp", "fbs", "restecg", "exang", "slope", "ca", "thal"]

target_column = "target"
original_target_column = "target_original"

print("Numeric features:", numeric_features)
print("Categorical features:", categorical_features)

Numeric features: ['age', 'trestbps', 'chol', 'thalach', 'oldpeak']
Categorical features: ['sex', 'cp', 'fbs', 'restecg', 'exang', 'slope', 'ca', 'thal']


## 3. Clean Data


In [ ]:
df = df_raw.copy()

# Impute missing values for Categorical by mode.
for col in ["ca", "thal"]:
    mode_value = df[col].mode(dropna=True)[0]
    missing_before = df[col].isna().sum()
    df[col] = df[col].fillna(mode_value)
    print(f"{col}: filled {missing_before} missing values with mode = {mode_value}")

# Covert categorical code from float to int to use one-hot encoder
for col in categorical_features:
    df[col] = df[col].astype(int)

# Keeping raw target to re-trace severity, then create binary target for classification.
df[original_target_column] = df[target_column].astype(int)
df[target_column] = (df[original_target_column] > 0).astype(int)

print("Missing values after cleaning:")
display(df.isna().sum().to_frame("missing_count"))

display(df.head())

ca: filled 4 missing values with mode = 0.0
thal: filled 2 missing values with mode = 3.0
Missing values after cleaning:


,missing_count
age,0
sex,0
cp,0
trestbps,0
chol,0
fbs,0
restecg,0
thalach,0
exang,0
oldpeak,0


,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,target,target_original
0,63.0,1,1,145.0,233.0,1,2,150.0,0,2.3,3,0,6,0,0
1,67.0,1,4,160.0,286.0,0,2,108.0,1,1.5,2,3,3,1,2
2,67.0,1,4,120.0,229.0,0,2,129.0,1,2.6,2,2,7,1,1
3,37.0,1,3,130.0,250.0,0,0,187.0,0,3.5,3,0,3,0,0
4,41.0,0,2,130.0,204.0,0,2,172.0,0,1.4,1,0,3,0,0


## 4. Train/Test Split

In [ ]:
# Split before fit scaler/encoder to avoid data leakage. `stratify=y` help keep class rate between train & test.
X = df[numeric_features + categorical_features]
y = df[target_column]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y,
)

print(f"X_train: {X_train.shape}")
print(f"X_test : {X_test.shape}")
print("\nTrain target distribution:")
display(y_train.value_counts(normalize=True).sort_index().to_frame("ratio"))
print("\nTest target distribution:")
display(y_test.value_counts(normalize=True).sort_index().to_frame("ratio"))

X_train: (242, 13)
X_test : (61, 13)

Train target distribution:


,ratio
target,
0,0.541322
1,0.458678



Test target distribution:


,ratio
target,
0,0.540984
1,0.459016


## 5. Scaling and One-Hot Encoding

In [ ]:
# Numeric features using StandardScaler
# Categorical features using OneHotEncoder => Preprocessor fit on train, then transform test.
preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", StandardScaler(), numeric_features),
        (
            "categorical",
            OneHotEncoder(handle_unknown="ignore", sparse_output=False),
            categorical_features,
        ),
    ],
    remainder="drop",
    verbose_feature_names_out=False,
)

# Fit on train, transform train/test
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

feature_names = preprocessor.get_feature_names_out().tolist()

X_train_processed_df = pd.DataFrame(X_train_processed, columns=feature_names)
X_test_processed_df = pd.DataFrame(X_test_processed, columns=feature_names)
y_train_df = y_train.reset_index(drop=True).to_frame(target_column)
y_test_df = y_test.reset_index(drop=True).to_frame(target_column)

print(f"Processed train shape: {X_train_processed_df.shape}")
print(f"Processed test shape : {X_test_processed_df.shape}")
display(X_train_processed_df.head())

Processed train shape: (242, 28)
Processed test shape : (61, 28)


,age,trestbps,chol,thalach,oldpeak,sex_0,sex_1,cp_1,cp_2,cp_3,...,slope_1,slope_2,slope_3,ca_0,ca_1,ca_2,ca_3,thal_3,thal_6,thal_7
0,-0.729485,-0.395692,0.458139,0.708371,-0.445445,0.0,1.0,0.0,0.0,0.0,...,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0
1,0.050166,-0.054513,0.230598,0.222495,-0.891627,0.0,1.0,0.0,1.0,0.0,...,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0
2,-0.061212,0.059213,0.723605,0.399178,-0.891627,1.0,0.0,0.0,1.0,0.0,...,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0
3,-0.061212,-1.305501,1.121803,0.266666,-0.891627,0.0,1.0,0.0,1.0,0.0,...,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0
4,0.272924,0.514117,-0.167601,-1.190962,-0.713154,1.0,0.0,0.0,0.0,0.0,...,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0


## 6. Save Processed Outputs

In [ ]:
# Input for 03_Training.ipynb
# Save X/y riêng to training script or read direct on notebook 
X_train_processed_df.to_csv(PROCESSED_DIR / "X_train.csv", index=False)
X_test_processed_df.to_csv(PROCESSED_DIR / "X_test.csv", index=False)
y_train_df.to_csv(PROCESSED_DIR / "y_train.csv", index=False)
y_test_df.to_csv(PROCESSED_DIR / "y_test.csv", index=False)

# Save concaternate version easy to manual check on notebook/report
train_processed = pd.concat([X_train_processed_df, y_train_df], axis=1)
test_processed = pd.concat([X_test_processed_df, y_test_df], axis=1)
train_processed.to_csv(PROCESSED_DIR / "train_processed.csv", index=False)
test_processed.to_csv(PROCESSED_DIR / "test_processed.csv", index=False)

# Saving cleaned data without encode to debug and comparation with EDA
# This file not input for model, only use to check preprocessing.
df.to_csv(PROCESSED_DIR / "heart_cleaned.csv", index=False)

# Saving feature list after encode
pd.Series(feature_names, name="feature").to_csv(PROCESSED_DIR / "feature_names.csv", index=False)

# Saving fitted preprocessor to predict new data
joblib.dump(preprocessor, MODELS_DIR / "preprocessor.joblib")

print("Saved processed files:")
for path in sorted(PROCESSED_DIR.glob("*.csv")):
    print(f"- {path.name}")
print(f"Saved preprocessor: {MODELS_DIR / 'preprocessor.joblib'}")

Saved processed files:
- X_test.csv
- X_train.csv
- feature_names.csv
- heart_cleaned.csv
- test_processed.csv
- train_processed.csv
- y_test.csv
- y_train.csv
Saved preprocessor: /home/dthanh/Documents/Personal_Doc/ThS/HP2/PPNCKH/Project/heart-disease-classification/models/preprocessor.joblib


## 7. Save Preprocessing Summary



In [ ]:
# File summary to write preprocessing decision to report
preprocessing_summary = {
    "raw_data_path": str(RAW_DATA_PATH),
    "n_rows": int(df.shape[0]),
    "n_original_columns": len(columns),
    "numeric_features": numeric_features,
    "categorical_features": categorical_features,
    "n_processed_features": len(feature_names),
    "target_column": target_column,
    "original_target_column": original_target_column,
    "target_distribution": df[target_column].value_counts().sort_index().to_dict(),
    "test_size": TEST_SIZE,
    "random_state": RANDOM_STATE,
    "split_strategy": "stratified by binary target",
    "missing_values_after_cleaning": df.isna().sum().to_dict(),
}

summary_path = PROCESSED_DIR / "preprocessing_summary.json"
summary_path.write_text(json.dumps(preprocessing_summary, indent=2), encoding="utf-8")

print(f"Saved preprocessing summary to: {summary_path}")
display(preprocessing_summary)

Saved preprocessing summary to: /home/dthanh/Documents/Personal_Doc/ThS/HP2/PPNCKH/Project/heart-disease-classification/data/processed/preprocessing_summary.json


{'raw_data_path': '/home/dthanh/Documents/Personal_Doc/ThS/HP2/PPNCKH/Project/heart-disease-classification/data/raw/heart.csv',
 'n_rows': 303,
 'n_original_columns': 14,
 'numeric_features': ['age', 'trestbps', 'chol', 'thalach', 'oldpeak'],
 'categorical_features': ['sex',
  'cp',
  'fbs',
  'restecg',
  'exang',
  'slope',
  'ca',
  'thal'],
 'n_processed_features': 28,
 'target_column': 'target',
 'original_target_column': 'target_original',
 'target_distribution': {0: 164, 1: 139},
 'test_size': 0.2,
 'random_state': 42,
 'split_strategy': 'stratified by binary target',
 'missing_values_after_cleaning': {'age': 0,
  'sex': 0,
  'cp': 0,
  'trestbps': 0,
  'chol': 0,
  'fbs': 0,
  'restecg': 0,
  'thalach': 0,
  'exang': 0,
  'oldpeak': 0,
  'slope': 0,
  'ca': 0,
  'thal': 0,
  'target': 0,
  'target_original': 0}}

## 8. Next Step

Sau notebook này, bước tiếp theo nên là `03_Training.ipynb`:

- Đọc `data/processed/X_train.csv` và `data/processed/y_train.csv` để fit model.
- Đọc `data/processed/X_test.csv` và `data/processed/y_test.csv` để đánh giá.
- Không fit scaler/encoder lại trên test set.
